# 2. Baseline comparison

**What does a planner achieve unaided?** That is the bar, and Forecast Value Added is defined against it — so if the benchmark is weak, every model looks good and the comparison table becomes flattery.

This notebook establishes two things:

1. Why **Step 4's seasonal naive cannot be reused here**, which is the easy mistake.
2. What the corrected benchmark actually scores, per horizon.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd

from app.services.container import Container
from ml.forecasting.baselines import SEASONAL_LAG_DAYS, attach_seasonal_reference
from ml.forecasting.config import load_forecast_config
from ml.forecasting.dataset import HorizonDataset, build_history, build_horizon_dataset
from ml.forecasting.evaluate import fva_table, metrics_by_horizon_bucket
from ml.forecasting.sampling import sample_series
from ml.forecasting.split import build_origin_split, slice_fold
from ml.forecasting.train import build_estimator, train_forecaster

repo = Container().data_repository
config = load_forecast_config().smoke()

## 1. Why Step 4's seasonal naive is the wrong benchmark

`SeasonalNaiveBaseline` blends `lag_364_units` with a rolling mean. In a *nowcast* that is exactly right: `lag_364` at date D is units at D−364, the same weekday a year earlier.

In a horizon dataset, `lag_364_units` is measured at the **origin**. For a row at horizon *h*, that is `364 + h` days before the date being forecast.

In [ ]:
origin = pd.Timestamp("2025-06-01")

print(f"{'h':>4}  {'target date':>12}  {'Step 4 lag_364 reads':>22}  {'gap to target':>14}  weekday match")
print("-" * 84)
for h in (1, 7, 30, 90):
    target = origin + pd.Timedelta(days=h)
    step4_reads = origin - pd.Timedelta(days=SEASONAL_LAG_DAYS)
    correct_reads = target - pd.Timedelta(days=SEASONAL_LAG_DAYS)
    gap = (target - step4_reads).days
    match = step4_reads.day_name() == target.day_name()
    print(f"{h:>4}  {target.date()!s:>12}  {step4_reads.date()!s:>22}  {gap:>14}  {match}")

print()
print(f"The correct reference for h=90 is {(origin + pd.Timedelta(days=90) - pd.Timedelta(days=364)).date()},")
print("which is the same weekday and the same point in the season as the target.")

Wrong weekday, wrong point in the season, and progressively more wrong as *h* grows. Step 4's fallback chain also includes `lag_1_units` — the illegal nowcast feature this whole step exists to avoid.

`HorizonSeasonalNaive` reads units at `target_date − 364` instead, attached by `attach_seasonal_reference`. That value is knowable at the origin for any horizon under 364 days, because it lies more than a year in the past.

## 2. Build the dataset and score the benchmarks

In [ ]:
sample = sample_series(repo, n_series=config.sampling.n_series, seed=config.sampling.seed)
history = build_history(repo, config, sample)
view = repo.as_of(pd.to_datetime(history["date"]).dt.date.max())

dataset = build_horizon_dataset(history, view, config, sample)
dataset = HorizonDataset(
    frame=attach_seasonal_reference(dataset.frame, history),
    feature_names=[*dataset.feature_names, "seasonal_reference"],
    excluded=dataset.excluded,
)
split = build_origin_split(dataset.frame, config)

print(dataset.describe())
print(split.describe())

In [ ]:
trained = {}
for name in ("horizon_naive", "horizon_seasonal_naive", "lightgbm"):
    trained[name] = train_forecaster(
        dataset, build_estimator(name, seed=42), config, split
    )
    metrics = trained[name].metrics["test"]
    print(f"{name:24s} WMAPE {metrics.wmape:6.1%}   bias {metrics.bias_pct:+6.1%}   "
          f"trained in {trained[name].train_seconds:.2f}s")

## 3. Accuracy by horizon

A single blended WMAPE describes no decision anyone makes. The interesting question is how each benchmark degrades as the horizon grows.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for name, model in trained.items():
    buckets = model.bucket_metrics
    ax.plot(
        list(buckets.keys()),
        [m.wmape for m in buckets.values()],
        marker="o",
        label=name,
    )

ax.axhline(0.35, linestyle="--", color="grey", label="irreducible noise floor (Step 4)")
ax.set_ylabel("WMAPE")
ax.set_xlabel("horizon bucket")
ax.set_title("Accuracy degrades with horizon - as it should")
ax.legend()
plt.tight_layout()
plt.show()

**The upward slope is the single strongest evidence that the origin/target join is correct.** If forecasting 90 days out were as accurate as forecasting tomorrow, the model would not be forecasting — it would be reading information from the target date it should not have. A flat line here is a leakage alarm, not a triumph.

The dashed line is Step 4's measured noise floor. Nothing should go below it.

## 4. Forecast Value Added

In [ ]:
benchmark = trained["horizon_seasonal_naive"].bucket_metrics
table = fva_table(trained["lightgbm"].bucket_metrics, benchmark, model_name="lightgbm")

display(table.style.format({
    "horizon_seasonal_naive_wmape": "{:.1%}",
    "lightgbm_wmape": "{:.1%}",
    "fva_pp": "{:+.1%}",
}))

weak = table[table["fva_pp"] <= 0]
if not weak.empty:
    print()
    print("Buckets where the model adds nothing over the benchmark:")
    print(weak[["bucket", "fva_pp"]].to_string(index=False))
    print("At those horizons the seasonal naive is the honest choice.")

FVA is reported in **percentage points, not as a ratio**. Against a ~35% noise floor a ratio compresses every result into a narrow band, and a genuine four-point improvement reads as a rounding error.

Breaking it out by bucket is the point. A model that adds six points at h=1–7 and nothing at h=57–90 should be used for the short horizon and replaced by the benchmark for the long one — a conclusion a blended figure would never surface.

---

## Findings

- Step 4's seasonal naive is **not** reusable: its reference is `364 + h` days before the target, and its fallback chain includes an illegal nowcast feature.
- Error grows with horizon for every candidate, which is the expected and required shape.
- FVA is reported per bucket, including any bucket where it is zero or negative.

### Next

`03_model_training.ipynb` — how the horizon dataset is built, and what the model learns from it.